# Lab 1.1. Reading files: the Spanish CSV and columnar storage

**Module 2: Session 1. ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

By the end of this notebook you should be able to:

1. Inspect a file at the byte level before loading it, and decide the reading arguments from that inspection.
2. Read a CSV with Spanish conventions (`;` separator, decimal comma, thousands point, `latin-1` encoding, `dd/mm/yyyy` dates) obtaining correct types.
3. Determine the granularity of a table and verify it with a programmatic check.
4. Identify sentinel values and convert them into explicit missing values.
5. Compare CSV and Parquet in size, read time and type preservation, and justify the choice.

**How to work.** Each exercise has three cells: statement, code gap and check.
The checks use `assert`: if nothing is printed but the final message, you got it right.
Exercises marked *extension* are optional.

**Before handing in:** Kernel > Restart & Run All. The notebook must run top to bottom with no errors.

*Note on names.* The data come from Spanish public sources, so the column names in the files
(`fecha`, `intensidad`, `vmed`...) are in Spanish and are kept as they arrive. Variables and new
columns we create are in English.

## 0. Environment

Record the versions you worked with (rule 5 of Rule et al., 2019).

In [ ]:
import sys, platform
import pandas as pd, numpy as np, matplotlib
print("Python     ", sys.version.split()[0], "|", platform.system())
for m in (pd, np, matplotlib):
    print(f"{m.__name__:<11}", m.__version__)

## 0. Setup

This cell locates the course folder wherever the notebook is running: on your own machine, in
Colab from the course repository, or in Colab from the shared Drive folder. Run it first and
check that the file listing appears.

In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                             # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):                   # local clone
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")

    root = Path("/content/acs-mod2")                   # 1. try the repository
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root

    from google.colab import drive                     # 2. fall back to Drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError(
        "Could not find the course folder. Either set REPO to the course repository, "
        f"or add a shortcut to the shared folder in My Drive as {DRIVE}.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
print("Course folder:", BASE)
print("Working folder:", WORK)
sorted(p.name for p in DATA.iterdir())

## Exercise 0. The project folder *(10 minutes)*

This is how project information arrives: a folder with a dozen files written by different
programs, for different readers, with different conventions. Before opening any of them with a
library you need to know what each one is, and **the extension is a label, not a guarantee**:
in practice, some files lie.

The `peek()` helper shows the first bytes of a file and gives a provisional verdict. You do not
need to understand its code now; do notice **why trying to decode as UTF-8 is not enough**:
null bytes are valid UTF-8 and are everywhere in binary files.

In [ ]:
def peek(path, n=64):
    """First bytes of a file and a provisional text/binary verdict."""
    b = Path(path).read_bytes()[:n]
    nulls = b.count(0)
    printable = sum(1 for c in b if 32 <= c < 127 or c in (9, 10, 13))
    verdict = "binary" if nulls > 0 or printable / max(len(b), 1) < 0.85 else "text"
    print(f"{Path(path).name:<26} {verdict:<8} {b[:44]!r}")

files = [
    "aforos_202509.csv", "pm_ubicaciones.csv", "imd_septiembre.xlsx",
    "intensidades_2024.xls", "municipios.geojson", "municipios_shp/municipios.shp",
    "municipios_shp/municipios.dbf", "municipios_shp/municipios.prj",
    "mdt25_madrid.tif", "PNOA_2020_0559.las", "estructura_rev07.ifc", "PZ07_20250915.dat",
]
for f in files:
    peek(DATA / f)

**With your partner, and without opening anything else:** for each file, fill in the dictionary
with text or binary. In the second dictionary, note which files lie about their extension and
what they really are. When you do not know what something is, write down what you **can
observe**: that is also a professional answer.

Before running the check, predict: how many extensions lie?

In [ ]:
diagnosis = {
    "aforos_202509.csv": "text",          # worked example
    "pm_ubicaciones.csv": "?",
    "imd_septiembre.xlsx": "?",
    "intensidades_2024.xls": "?",
    "municipios.geojson": "?",
    "municipios.shp": "?",
    "municipios.dbf": "?",
    "municipios.prj": "?",
    "mdt25_madrid.tif": "?",
    "PNOA_2020_0559.las": "?",
    "estructura_rev07.ifc": "?",
    "PZ07_20250915.dat": "?",
}
# YOUR CODE HERE
raise NotImplementedError

# Which files lie about their extension, and what are they really?
# Accepted values: "html", "csv", "delimited text", "json", "xml"
lying = {
    # "file_name": "what it really is",
}
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
key = {
    "aforos_202509.csv": "text", "pm_ubicaciones.csv": "text",
    "imd_septiembre.xlsx": "binary", "intensidades_2024.xls": "text",
    "municipios.geojson": "text", "municipios.shp": "binary",
    "municipios.dbf": "binary", "municipios.prj": "text",
    "mdt25_madrid.tif": "binary", "PNOA_2020_0559.las": "binary",
    "estructura_rev07.ifc": "text", "PZ07_20250915.dat": "text",
}
wrong = [k for k in key if diagnosis.get(k) != key[k]]
assert not wrong, f"Revisit text/binary for: {wrong}"
assert set(lying) == {"intensidades_2024.xls", "PZ07_20250915.dat"}, (
    "The two liars are an .xls and a .dat. Look at their first bytes again.")
assert lying["intensidades_2024.xls"] == "html", "A file that starts with <html> is not Excel"
assert lying["PZ07_20250915.dat"] in ("csv", "delimited text"), (
    "A .dat with quotes and commas is delimited text, with four header lines")
print("Checks passed: 12 files classified, 2 extensions unmasked.")

**Debrief.** Three things to take from these ten minutes:

1. `intensidades_2024.xls` is HTML. This is the usual export of many public portals: the server
   builds a web table and serves it with an `.xls` extension so that Excel opens it. `read_excel`
   fails on it; `read_html` works. The extension decided nothing.
2. Almost nobody in the room knows `estructura_rev07.ifc`, and nobody needs to: it is text, it
   starts with `ISO-10303-21`, and that string identifies the standard in thirty seconds of
   searching. That is the professional answer to an unknown format: *I do not know it, but it is
   text, it says ISO 10303, and I know how to find out.*
3. A shapefile is several files and not all of them are binary: the `.prj` is WKT text and can
   be read in an editor. You will see this in Lab 1.2.

The rest of the session explains why these twelve files reduce to a handful of structures.

## 1. Before `pandas`: look at the bytes

A file with a `.csv` extension is under no obligation to contain comma-separated values.
The extension is a convention, not a guarantee. Before loading anything, look at the raw content.

`repr()` shows the special characters (`\\n`, `\\t`) that `print()` would interpret.

In [ ]:
counts_path = DATA / "aforos_202509.csv"
print("Size:", round(counts_path.stat().st_size / 1e6, 2), "MB")

with open(counts_path, "r", encoding="latin-1") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i >= 3:
            break

**Notice three things** before going on:

- the field separator is not the comma,
- decimals are written with a comma,
- the date is in day/month/year order.

`pandas` cannot guess any of the three.

### Exercise 1. The naive read

Read the file with `pd.read_csv()` with no argument other than the encoding, and store it in `naive`.
It will not raise an error. That is the problem: the failure is silent.

Look at both the columns and the **index** of the result before answering.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print("Columns detected:", naive.shape[1])
print("Column name:", list(naive.columns))
print("Index:", list(naive.index[:2]))
naive.head(3)

In [ ]:
assert naive.shape[1] == 1, f"Expected a single column, got {naive.shape[1]}"
assert "intensidad" not in naive.columns, "pandas has not split the fields"
assert not isinstance(naive.index, pd.RangeIndex), (
    "Part of the data should have ended up in the index")
print("Checks passed: the naive read does not split the fields")
print("and also pushes part of each row into the index.")

_Answer (edit this cell):_ it detected ___ column(s). The default separator of `read_csv` is the
comma. The header line contains none, so `pandas` infers a single column. The data lines do
contain commas, the decimal ones, so they have more fields than the header, and `pandas` resolves
the mismatch by placing the extra values in ...

### Exercise 2. The correct read

Now read the file into `counts` with the arguments needed so that:

- the six columns are separated,
- `intensidad` is an integer,
- `ocupacion` and `vmed` are floats,
- `fecha` is a datetime, with the day before the month,
- accented characters are read correctly.

Arguments you will need: `sep`, `decimal`, `thousands`, `encoding`, `parse_dates`, `dayfirst`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(counts.dtypes)
counts.head()

In [ ]:
assert counts.shape == (21568, 6), f"Expected (21568, 6), got {counts.shape}"
assert counts["fecha"].dtype.kind == "M", "The fecha column is not a datetime"
assert counts["intensidad"].dtype.kind in "iu", "intensidad should be an integer"
assert counts["ocupacion"].dtype.kind == "f", "ocupacion should be a float"
assert counts["fecha"].min() == pd.Timestamp("2025-09-01"), (
    "The minimum date is not 1 September: check dayfirst")
assert counts["intensidad"].max() > 1000, (
    "If the maximum intensity is below 1000, the thousands point was misread")
print("Checks passed.")

> **Why `dayfirst` matters.** Without it, `pandas` cannot parse `01/09/2025 00:00:00` and leaves
> the column as text, raising nothing. Check it in the next cell. Older versions of the library
> behaved differently: they silently converted month-first, so 1 September became 9 January.
> The rule is not to memorise what each version does, but to always declare the file's convention.

In [ ]:
no_dayfirst = pd.read_csv(counts_path, sep=";", decimal=",", thousands=".",
                          encoding="latin-1", parse_dates=["fecha"])
print("dtype of fecha without dayfirst:", no_dayfirst["fecha"].dtype)
print("left as text?                 :", no_dayfirst["fecha"].dtype.kind != "M")
print()
print("The dangerous part comes now: min() and max() still work,")
print("but they compare strings, not dates.")
print("  min:", no_dayfirst["fecha"].min())
print("  max:", no_dayfirst["fecha"].max())
print()
print("Both look like the right answer. That is exactly the trap.")

_Answer:_ at which later point of the analysis would this error surface?

### Exercise 3. Granularity

The **granularity** of a table is what one row represents (Lau, Gonzalez and Nolan, 2023, ch. 9).
Stating it is not enough: it has to be checked.

If each row were one hourly reading of one measurement point, the pair `(id, fecha)` would be
unique. Check whether it is. Store in `n_duplicates` the number of duplicated rows considering
all columns, and in `dups` the affected rows.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print("Duplicated rows:", n_duplicates)
print("Unique (id, fecha) pairs:", not counts.duplicated(subset=["id", "fecha"]).any())
dups.head(4)

In [ ]:
assert n_duplicates == 40, f"Expected 40 duplicates, got {n_duplicates}"
assert len(dups) == 80, "dups should contain both copies of each duplicate"
print("Checks passed: the declared granularity does not hold.")

The duplicates come from retransmissions of the field equipment. They appear regularly in real
traffic files. Remove them, keeping the first occurrence.

In [ ]:
counts = counts.drop_duplicates().reset_index(drop=True)
assert len(counts) == 21528
print("Rows after removing duplicates:", len(counts))

### Exercise 4. Sentinel values

The notes sheet of the source file says that `-1` means no measurement. A sentinel is an
impossible value used to encode a gap: `-1`, `-9999`, `-99.99`, `NA`, the empty string.
If it is not translated to `NaN`, it enters the means and contaminates them.

Count how many sentinel values there are in `vmed`, store it in `n_sentinel`, and replace them
with `np.nan`. Compare the mean before and after.

In [ ]:
mean_before = counts["vmed"].mean()
# YOUR CODE HERE
raise NotImplementedError
mean_after = counts["vmed"].mean()
print(f"Sentinels: {n_sentinel}")
print(f"Mean before: {mean_before:.2f} km/h")
print(f"Mean after : {mean_after:.2f} km/h")
print(f"Bias introduced: {mean_after - mean_before:.2f} km/h")

In [ ]:
assert n_sentinel > 2000, "Expected more than 2000 sentinels"
assert counts["vmed"].isna().sum() == n_sentinel
assert (counts["vmed"].dropna() > 0).all(), "Some negative value remains in vmed"
assert mean_after - mean_before > 5, "The bias should be several km/h"
print("Checks passed.")

> **Why this is not a detail.** Sentinels were invented when formats could not represent missing
> values. They stay in files for compatibility. The practical consequence is that a mean computed
> without translating them is wrong and nobody notices, because the result has the right order
> of magnitude.

### Exercise 4b. Where are the gaps? *(extension)*

Knowing how many values are missing is not the same as knowing **why**. Two files with the same
number of gaps demand opposite decisions: if they are concentrated in three points, you have
three broken sensors and you exclude them; if they are spread evenly across all of them, the
cause is systemic and excluding points fixes nothing.

Build `gaps_per_point`, the number of gaps in `vmed` for each `id`, and `gap_fraction`, that
number divided by the readings each point contributes. Then look at the spread.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
print(f"Points with at least one gap: {len(gaps_per_point)} of {counts['id'].nunique()}")
print(f"Gap fraction: min {gap_fraction.min():.1%}, max {gap_fraction.max():.1%}")
gap_fraction.sort_values(ascending=False).head(5).round(3)

In [ ]:
assert len(gaps_per_point) == 30, (
    f"Expected all 30 points to be affected, got {len(gaps_per_point)}")
assert gap_fraction.max() - gap_fraction.min() < 0.06, (
    "The gaps look concentrated in a few points: check that you divided by the "
    "readings of each point and not by the total")
print("Checks passed: every point loses a similar share of its readings.")

_Answer:_ no point stands out. The gaps are not broken sensors, they are systemic, and dropping
points would remove good data without removing the problem. Note what you can **not** conclude
from this: an even spread across points says nothing about the spread across *hours*. A sensor
that fails every night at the same time gives exactly this picture, and its gaps are anything
but random. You check that in Session 2.

### Exercise 5. CSV versus Parquet *(run on the projector)*

In class you voted how much smaller the Parquet would be: 2, 10 or 50 times. Now it is measured.

CSV is delimited text and stores the table row by row. Parquet is binary, stores the table
column by column and carries the schema inside. Check the three consequences:
size, read time and type preservation.

Save `counts` as `counts.parquet` in the working folder and complete the measurements.

In [ ]:
import time
parquet_path = WORK / "counts.parquet"
# YOUR CODE HERE
raise NotImplementedError

size_csv = counts_path.stat().st_size / 1e6
size_pq = parquet_path.stat().st_size / 1e6

t0 = time.perf_counter(); _ = pd.read_csv(counts_path, sep=";", decimal=",", thousands=".",
                                          encoding="latin-1", parse_dates=["fecha"], dayfirst=True)
t_csv = time.perf_counter() - t0
t0 = time.perf_counter(); reloaded = pd.read_parquet(parquet_path)
t_pq = time.perf_counter() - t0

print(f"CSV     {size_csv:6.2f} MB   {t_csv*1000:6.0f} ms")
print(f"Parquet {size_pq:6.2f} MB   {t_pq*1000:6.0f} ms")
print(f"Size ratio: {size_csv/size_pq:.1f}x")

In [ ]:
assert size_pq < size_csv, "The Parquet should be smaller"
assert reloaded["fecha"].dtype.kind == "M", (
    "After the round trip the date should still be a date")
assert reloaded.dtypes.equals(counts.dtypes), "Types were not preserved"
print("Checks passed: Parquet keeps the schema; the CSV loses it.")

Selective column reads. A CSV must be read whole even if only two fields are needed;
a columnar format need not.

In [ ]:
two_cols = pd.read_parquet(parquet_path, columns=["id", "intensidad"])
print(two_cols.shape, "->", list(two_cols.columns))

**The honest counterexample.** Parquet is not always smaller. Its schema and footer have a fixed
cost that is only amortised with volume. Check it with the locations table, which has 30 rows.

In [ ]:
loc_tmp = pd.read_csv(DATA / "pm_ubicaciones.csv", sep=";", decimal=",",
                      thousands=".", encoding="latin-1")
loc_tmp.to_parquet(WORK / "loc_tmp.parquet", index=False)
b_csv = (DATA / "pm_ubicaciones.csv").stat().st_size
b_pq = (WORK / "loc_tmp.parquet").stat().st_size
print(f"30 rows:  CSV {b_csv} bytes  |  Parquet {b_pq} bytes  ->  Parquet {b_pq / b_csv:.1f}x LARGER")
print("Columnar formats pay off at scale, not on thirty-row tables.")

_Answer:_ which column do you think accounts for most of the space saving, and why?

### Exercise 6. The question of the session (first half)

> At what elevation are the ten busiest traffic counters in Madrid?

This half solves the tabular part. Compute the mean hourly intensity of each measurement point,
keep the ten largest, join them with the locations table and save the result as `top10.parquet`,
which is the input of Lab 1.2.

The locations table is `pm_ubicaciones.csv`, with the same Spanish conventions.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
top10

In [ ]:
assert len(top10) == 10, f"Expected 10 rows, got {len(top10)}"
assert {"x_utm", "y_utm", "nombre"} <= set(top10.columns), (
    "A column from the locations table is missing: check the merge")
assert top10["x_utm"].notna().all(), "Some locations have no coordinate: the merge failed"
assert top10["mean_intensity"].is_monotonic_decreasing, "Sort from largest to smallest"
assert top10["x_utm"].between(432000, 452000).all(), "Coordinates outside the working area"
print("Checks passed.")

In [ ]:
top10.to_parquet(WORK / "top10.parquet", index=False)
print("Saved", WORK / "top10.parquet")
print("Lab 1.2 rebuilds it by itself if this session has expired, so nothing is lost.")

### Extension A. A public-body Excel

Excel is not on today's syllabus, but it is the format in which a good part of project
information arrives. `imd_septiembre.xlsx` has the usual shape of a sheet published by an
administration: a title, a subtitle, a row of merged headers and the real header further down,
plus a separate notes sheet.

First see which sheets it has with `sheet_name=None`. Then read the `Datos` sheet into `imd`
skipping the header rows, so that the columns are `id`, `nombre`, `IMD (veh/día)`,
`Ocupación (%)` and `V media (km/h)`.

In [ ]:
sheets = pd.read_excel(DATA / "imd_septiembre.xlsx", sheet_name=None)
print("Sheets:", list(sheets))
sheets["Datos"].head(6)

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
imd.head()

In [ ]:
assert list(imd.columns)[:2] == ["id", "nombre"], f"Wrong header: {list(imd.columns)}"
assert imd.shape == (30, 5), f"Expected (30, 5), got {imd.shape}"
assert imd["IMD (veh/día)"].dtype.kind in "iuf"
print("Checks passed.")

### Extension B. Hourly profile

Plot the mean intensity by hour of day, separating weekdays from weekends.
Use the `.dt` accessor to extract the hour and the day of the week.

In [ ]:
import matplotlib.pyplot as plt
# YOUR CODE HERE
raise NotImplementedError


### Extension C. Provenance

Rule et al. (2019), rule 8: data are shared with their description and origin. Add to `top10`
three provenance columns (`source_file`, `extraction_date`, `n_source_rows`) and save it again.
In session 2 you will see why they are needed.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError
top10_prov.head(2)

---

## Take-aways

- A file's extension is a convention, not a guarantee: inspect before loading.
- A CSV does not carry its schema. The `read_csv` arguments are that schema, written by hand.
- Granularity is declared and checked; never assumed.
- An untranslated sentinel produces wrong means without raising any exception.
- Parquet keeps types, is smaller at scale and allows reading single columns. For any table you
  will re-read, it is the default choice.

**References.** Lau, Gonzalez and Nolan (2023), ch. 8 and 9. Wickham (2014). Rule et al. (2019).